# Modal gradients & differentiable inverse design

> Compute exact gradients of effective indices and EME objectives w.r.t. design
> parameters, and drive a differentiable level-set design - all on top of the
> ordinary (non-differentiable) mode solver.

meow's FDE backends solve each cross-section with an external eigensolver, so
meow does not differentiate *through* a mode solve directly. Instead it supplies
the **modal adjoint**: first-order perturbation theory gives the exact
sensitivity of a mode's effective index to the permittivity, evaluated from the
primal fields,

$$\mathrm{d}(n_\mathrm{eff}) = \frac{c\,\varepsilon_0}{4P}\int \mathrm{d}\varepsilon\;|\mathbf{E}|^2\,\mathrm{d}A .$$

This notebook shows the three layers:

1. `meow.neff_sensitivity` / `neff_gradient` - the adjoint kernel (this page);
2. `meow.make_differentiable_neffs` - a `jax.custom_vjp` so `jax.grad` flows
   through the solve into the JAX/SAX EME cascade;
3. `meow.levelset` - a smooth density -> permittivity map for inverse design.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import meow as mw
from meow import levelset

jax.config.update("jax_enable_x64", True)
WL = 1.55

## 1. Modal sensitivity (the adjoint kernel)

We solve the fundamental mode of a silicon strip waveguide, map *where* its
effective index is sensitive to an index change, and validate the kernel against
finite differences.

In [ ]:
def cross_section(width, n_core=3.45, n_clad=1.444, npx=121, npy=81):
    core = mw.Structure(
        material=mw.IndexMaterial(name="core", n=n_core),
        geometry=mw.Box(x_min=-width / 2, x_max=width / 2, y_min=0.0, y_max=0.22,
                        z_min=-1.0, z_max=2.0),
        mesh_order=5,
    )
    clad = mw.Structure(
        material=mw.IndexMaterial(name="clad", n=n_clad),
        geometry=mw.Box(x_min=-1.5, x_max=1.5, y_min=-1.0, y_max=1.0,
                        z_min=-1.0, z_max=2.0),
        mesh_order=10,
    )
    mesh = mw.Mesh2D(x=np.linspace(-1.5, 1.5, npx), y=np.linspace(-1.0, 1.0, npy))
    cell = mw.create_cells([core, clad], mesh, [1.0], z_min=0.0)[0]
    return mw.CrossSection.from_cell(cell=cell, env=mw.Environment(wl=WL))


cs = cross_section(0.5)
mode = mw.compute_modes(cs, num_modes=1)[0]
print(f"neff = {float(np.real(mode.neff)):.4f},  modal power P = {mw.modal_power(mode):.3f}")

# d(neff)/d(eps_xx) sensitivity density - where neff "cares" about the index
s_xx, s_yy, s_zz = mw.neff_sensitivity(mode)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
extent = [cs.mesh.x_.min(), cs.mesh.x_.max(), cs.mesh.y_.min(), cs.mesh.y_.max()]
im = ax1.imshow(np.real(cs.nx).T, origin="lower", extent=extent, cmap="gray_r")
ax1.set(title="refractive index $n_x$", xlabel="x [um]", ylabel="y [um]")
fig.colorbar(im, ax=ax1, shrink=0.8)
im2 = ax2.imshow(s_xx.T, origin="lower", extent=extent, cmap="magma")
ax2.set(title="sensitivity density  d(neff)/d(eps_xx)",
        xlabel="x [um]", ylabel="y [um]")
fig.colorbar(im2, ax=ax2, shrink=0.8)
fig.tight_layout()

The sensitivity peaks in the core where the field is strongest - a coarse index
change there moves $n_\mathrm{eff}$ most. Now validate the kernel: scaling every
index by $(1+s)$ has $\mathrm{d}\varepsilon/\mathrm{d}s = 2\varepsilon$, and the
adjoint must match finite differences.

In [ ]:
grad = mw.neff_gradient(
    mode, 2 * np.asarray(cs.nx) ** 2, 2 * np.asarray(cs.ny) ** 2,
    2 * np.asarray(cs.nz) ** 2,
)


def neff_scaled(t):
    # scale every index by (1 + t) and re-solve
    c = cross_section(0.5, n_core=3.45 * (1 + t), n_clad=1.444 * (1 + t))
    return mw.compute_modes(c, num_modes=1)[0].neff


fd = mw.finite_difference_gradient(neff_scaled, step=1e-3)
print(f"adjoint   d(neff)/ds = {grad.real:.5f}")
print(f"finite-diff          = {fd.real:.5f}")
print(f"relative error       = {abs(grad - fd) / abs(fd):.1e}")

## 2. End-to-end autodiff through the EME cascade

`make_differentiable_neffs` wraps the (non-differentiable) solve as a
`jax.custom_vjp`: the forward runs the solve, the backward uses the adjoint
kernel with a cheap permittivity Jacobian (no extra eigensolve). The returned
effective indices compose with the already-JAX SAX cascade, so `jax.grad` of any
objective flows back to the design parameters. Here we **optimize a waveguide
width to hit a target effective index** by gradient descent.

In [ ]:
def solve(params):
    return [mw.compute_modes(cross_section(float(params[0])), num_modes=1)]


def cross_sections(params):
    return [cross_section(float(params[0]))]


neffs = mw.make_differentiable_neffs(solve, shape=(1, 1), cross_sections=cross_sections)

TARGET = 2.55


@jax.jit
def objective(params):
    return (jnp.real(neffs(params)[0, 0]) - TARGET) ** 2


value_and_grad = jax.value_and_grad(objective)

w = jnp.array([0.40])
lr, history = 0.4, []
for _ in range(12):
    neff_now = float(jnp.real(neffs(w)[0, 0]))  # record the actual index
    _loss, g = value_and_grad(w)
    history.append((float(w[0]), neff_now))
    w = jnp.clip(w - lr * g, 0.25, 1.1)  # keep the width physical
history = np.array(history)
print(f"start width {history[0,0]:.3f} um -> width {float(w[0]):.3f} um")
print(f"neff {history[0,1]:.4f} -> {float(jnp.real(neffs(w)[0,0])):.4f} (target {TARGET})")

In [ ]:
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4))
axa.plot(history[:, 0], "o-")
axa.set(xlabel="iteration", ylabel="width [um]", title="design parameter")
axb.axhline(TARGET, color="k", ls="--", label="target")
axb.plot(history[:, 1], "o-", color="C3", label="neff")
axb.set(xlabel="iteration", ylabel="neff", title="gradient descent to target neff")
axb.legend()
fig.tight_layout()

A few exact-gradient steps land the width on the target index. The gradient came
through the mode solve via the modal adjoint - no finite differencing, and the
cost is one eigensolve per value-and-gradient regardless of parameter count.

## 3. Differentiable inverse design with level sets

A staircased polygon boundary is not differentiable in geometry; a **density /
level-set** field is. `meow.levelset` maps a smooth density $\rho(x,y)\in[0,1]$
to permittivity ($\varepsilon = \varepsilon_\mathrm{clad} + \rho\,
(\varepsilon_\mathrm{core}-\varepsilon_\mathrm{clad})$) and provides the analytic
Jacobian, so the whole chain *params -> density -> eps -> modes -> objective* is
differentiable. Here a soft-rectangle width is optimized to a target index.

In [ ]:
N_CORE, N_CLAD, BETA = 3.45, 1.444, 40.0
Y0, HALF_H = 0.11, 0.11
DMESH = mw.Mesh2D(x=np.linspace(-1.5, 1.5, 121), y=np.linspace(-1.0, 1.0, 81))


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def density(half_w):
    def rho(x, y):
        return sigmoid(BETA * (half_w - np.abs(x))) * sigmoid(BETA * (HALF_H - np.abs(y - Y0)))
    return rho


def ddensity(half_w):
    def drho(x, y):
        sx = sigmoid(BETA * (half_w - np.abs(x)))
        return BETA * sx * (1 - sx) * sigmoid(BETA * (HALF_H - np.abs(y - Y0)))
    return drho


def ls_solve(params):
    cs = levelset.density_cross_section(
        DMESH, mw.Environment(wl=WL), density(float(params[0])), n_min=N_CLAD, n_max=N_CORE
    )
    return [mw.compute_modes(cs, num_modes=1)]


def ls_eps_jac(params, j):
    d = levelset.eps_jacobian_components(DMESH, ddensity(float(params[0])), N_CLAD, N_CORE)
    return [d]


ls_neffs = mw.make_differentiable_neffs(ls_solve, shape=(1, 1), eps_jacobian=ls_eps_jac)
g = float(jax.grad(lambda p: jnp.real(ls_neffs(p)[0, 0]))(jnp.array([0.25]))[0])
print(f"d(neff)/d(half_width) via differentiable level set = {g:.3f} (>0: wider -> higher neff)")

In [ ]:
# visualize the density-defined waveguide the gradient is taken through
rho = density(0.25)
RHO = rho(np.asarray(DMESH.Xz), np.asarray(DMESH.Yz))
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(RHO.T, origin="lower", cmap="viridis",
               extent=[-1.5, 1.5, -1.0, 1.0], vmin=0, vmax=1)
ax.set(title="level-set density rho(x, y)", xlabel="x [um]", ylabel="y [um]")
fig.colorbar(im, ax=ax, shrink=0.8, label="rho")
fig.tight_layout()

## 4. Scaling these on a cluster

The gradient and primal evaluations are independent per frequency, so a
broadband objective parallelizes as a reduction across nodes. A few knobs help:

- **Thread pinning** - cap each solve's BLAS threads so co-scheduled jobs don't
  oversubscribe a node:
  ```python
  import os; os.environ["MEOW_SOLVER_THREADS"] = "4"      # or:
  with mw.limit_threads(4):
      modes = mw.compute_modes(cs, num_modes=8)
  ```
- **Frequency-distributed spectra** - spread a dense sweep across nodes and
  cascade the per-wavelength S-matrices concurrently:
  ```python
  S = mw.compute_s_matrix_spectrum(cells, env, wls=wls,
                                   wls_per_job=8, cascade_workers=8,
                                   executor=mw.slurm_executor(...))
  ```
- **Cached smoothing** - the (wavelength-independent) subpixel-smoothing geometry
  is computed once per cell and reused across a sweep automatically.
- **Sparse solves for huge grids** - `meow.fde.sparse.scalar_neffs(cs)` uses a
  sparse shift-invert solve that scales with grid nonzeros, not a dense matrix.

```python
from meow.fde import sparse
neff = sparse.scalar_neffs(cs, num_modes=1)[0]
```

## Summary

- `neff_sensitivity` / `neff_gradient` - exact modal adjoint from the primal
  fields (validated to ~1e-8).
- `neff_value_and_grad` - effective indices + their Jacobian from one solve.
- `make_differentiable_neffs` - `jax.custom_vjp` so `jax.grad` flows through the
  solve and the JAX/SAX cascade.
- `meow.levelset` - smooth density -> permittivity for differentiable inverse
  design.